# Denoising Autoencoder on MNIST

**Assignment:** Build a deep learning model that removes noise from images using a Convolutional Autoencoder trained on MNIST.

**Approach:**
- Two-phase training: clean reconstruction warmup → denoising fine-tune
- Architecture: 3-block Convolutional Encoder + 3-block Transposed-Conv Decoder
- Noise Model: Additive Gaussian Noise (σ=0.4), clamped to [0,1]

---

## 1. Imports & Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch version : {torch.__version__}')
print(f'Device          : {device}')

## 2. Load & Preprocess MNIST

In [ ]:
# Transform: convert PIL images to float tensors in [0, 1]
transform = transforms.Compose([transforms.ToTensor()])

# Load MNIST — downloads automatically if not cached
train_data = datasets.MNIST('data', train=True,  download=True, transform=transform)
test_data  = datasets.MNIST('data', train=False, download=True, transform=transform)

print(f'Train samples : {len(train_data):,}')
print(f'Test samples  : {len(test_data):,}')
print(f'Image shape   : {train_data[0][0].shape}  (C x H x W)')
print(f'Pixel range   : [0, 1] — normalized by ToTensor()')

# For a quick experiment: use a 10k subset for training, full 10k test set
# (Replace with full train_data for best results if compute allows)
np.random.seed(SEED)
idx = np.random.choice(len(train_data), 12000, replace=False)
train_subset = Subset(train_data, idx[:10000])
val_subset   = Subset(train_data, idx[10000:])

BATCH_SIZE = 256
train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_subset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_data,    batch_size=BATCH_SIZE, shuffle=False)

print(f'\nTrain batches : {len(train_loader)}')
print(f'Val batches   : {len(val_loader)}')
print(f'Test batches  : {len(test_loader)}')

In [ ]:
# Visualize a batch of clean training images
imgs, labels = next(iter(train_loader))

fig, axes = plt.subplots(2, 10, figsize=(16, 3.5))
fig.suptitle('Sample Training Images (Clean)', fontsize=13, fontweight='bold')
for i in range(10):
    axes[0, i].imshow(imgs[i].squeeze(), cmap='gray')
    axes[0, i].set_title(labels[i].item(), fontsize=10)
    axes[0, i].axis('off')
    axes[1, i].imshow(imgs[i+10].squeeze(), cmap='gray')
    axes[1, i].set_title(labels[i+10].item(), fontsize=10)
    axes[1, i].axis('off')
plt.tight_layout()
plt.show()

## 3. Noise Generation

In [ ]:
def add_gaussian_noise(imgs: torch.Tensor, noise_factor: float = 0.4) -> torch.Tensor:
    """
    Add Additive White Gaussian Noise (AWGN) to a batch of images.
    
    Args:
        imgs         : Clean images tensor in [0, 1]  — shape (B, C, H, W)
        noise_factor : Standard deviation of the Gaussian noise  (σ)
    
    Returns:
        Noisy images clamped to [0, 1]
    """
    noise = noise_factor * torch.randn_like(imgs)
    return torch.clamp(imgs + noise, min=0.0, max=1.0)


NOISE_FACTOR = 0.4   # σ = 0.4 → moderately severe corruption

# Visualize clean vs. noisy
sample_imgs = imgs[:10]
noisy_imgs  = add_gaussian_noise(sample_imgs, NOISE_FACTOR)

print(f'Clean pixel range : [{sample_imgs.min():.3f}, {sample_imgs.max():.3f}]')
print(f'Noisy pixel range : [{noisy_imgs.min():.3f}, {noisy_imgs.max():.3f}]')

fig, axes = plt.subplots(2, 10, figsize=(16, 3.5))
fig.suptitle(f'Clean (top) vs Noisy (bottom) — σ = {NOISE_FACTOR}', fontsize=13, fontweight='bold')
for i in range(10):
    axes[0, i].imshow(sample_imgs[i].squeeze(), cmap='gray', vmin=0, vmax=1)
    axes[0, i].axis('off')
    axes[1, i].imshow(noisy_imgs[i].squeeze(), cmap='gray', vmin=0, vmax=1)
    axes[1, i].axis('off')
axes[0, 0].set_ylabel('Clean', fontsize=10, rotation=90, labelpad=35, va='center')
axes[1, 0].set_ylabel('Noisy', fontsize=10, rotation=90, labelpad=35, va='center')
plt.tight_layout()
plt.show()

## 4. Model Architecture — Convolutional Denoising Autoencoder

In [ ]:
class DenoisingAutoencoder(nn.Module):
    """
    Convolutional Denoising Autoencoder (DAE)
    
    Encoder
    -------
    Block 1: Conv(1→32, 3×3) → ReLU → MaxPool(2×2)     [28×28 → 14×14]
    Block 2: Conv(32→16, 3×3) → ReLU → MaxPool(2×2)    [14×14 →  7×7]
    Block 3: Conv(16→8, 3×3)  → ReLU                   [7×7  (bottleneck: 8×7×7 = 392 features)]

    Decoder
    -------
    Block 1: ConvTranspose(8→16, 2×2, stride=2) → ReLU  [ 7×7 → 14×14]
    Block 2: ConvTranspose(16→32, 2×2, stride=2) → ReLU [14×14 → 28×28]
    Block 3: Conv(32→1, 3×3) → Sigmoid                  [28×28 — output in [0,1]]

    Design choices:
    - MaxPool in encoder compresses spatial resolution aggressively → forces the model
      to learn a compact, noise-invariant representation.
    - ConvTranspose in decoder mirrors the encoder symmetrically → good gradient flow.
    - Sigmoid output ensures values match the [0,1] target pixel range.
    - Lightweight (9K params) → trains efficiently even on CPU.
    """

    def __init__(self):
        super().__init__()

        # ---- Encoder ----
        self.enc1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2)                           # 28×28 → 14×14
        )
        self.enc2 = nn.Sequential(
            nn.Conv2d(32, 16, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2)                           # 14×14 → 7×7
        )
        self.enc3 = nn.Sequential(
            nn.Conv2d(16, 8, kernel_size=3, padding=1),  # 7×7 bottleneck
            nn.ReLU(inplace=True)
        )

        # ---- Decoder ----
        self.dec1 = nn.Sequential(
            nn.ConvTranspose2d(8, 16, kernel_size=2, stride=2),   # 7×7 → 14×14
            nn.ReLU(inplace=True)
        )
        self.dec2 = nn.Sequential(
            nn.ConvTranspose2d(16, 32, kernel_size=2, stride=2),  # 14×14 → 28×28
            nn.ReLU(inplace=True)
        )
        self.dec3 = nn.Conv2d(32, 1, kernel_size=3, padding=1)    # 28×28 → 28×28

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Encode
        x = self.enc1(x)
        x = self.enc2(x)
        x = self.enc3(x)     # bottleneck: (B, 8, 7, 7)
        # Decode
        x = self.dec1(x)
        x = self.dec2(x)
        x = torch.sigmoid(self.dec3(x))
        return x


model = DenoisingAutoencoder().to(device)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters     : {total_params:,}')
print(f'Trainable parameters : {trainable_params:,}')
print()
print(model)

## 5. Training — Two-Phase Strategy

**Phase 1 — Clean Reconstruction Warmup**  
Train the autoencoder first on clean images (input = target). This initializes the encoder/decoder weights to represent digit structure before introducing noise.

**Phase 2 — Denoising Fine-Tune**  
Switch to the DAE objective: input = noisy images, target = clean images.

In [ ]:
import time

criterion = nn.MSELoss()   # MSE is the standard loss for pixel-level reconstruction

# ── Phase 1: clean reconstruction warmup ──────────────────────────────────────
optimizer_p1 = torch.optim.Adam(model.parameters(), lr=1e-3)

WARMUP_EPOCHS = 10
print(f'Phase 1: Clean Reconstruction Warmup ({WARMUP_EPOCHS} epochs)')
print('-' * 55)

t0 = time.time()
for epoch in range(1, WARMUP_EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for imgs, _ in train_loader:
        imgs = imgs.to(device)
        optimizer_p1.zero_grad()
        output = model(imgs)                        # clean → clean
        loss = criterion(output, imgs)
        loss.backward()
        optimizer_p1.step()
        total_loss += loss.item() * imgs.size(0)
    avg_loss = total_loss / len(train_subset)
    if epoch % 5 == 0 or epoch == 1:
        print(f'  Epoch [{epoch:2d}/{WARMUP_EPOCHS}] Clean Loss: {avg_loss:.5f}')

print(f'  Warmup complete in {time.time()-t0:.1f}s')

In [ ]:
# ── Phase 2: Denoising fine-tune ──────────────────────────────────────────────
optimizer_p2  = torch.optim.Adam(model.parameters(), lr=5e-4)
scheduler     = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer_p2, patience=3, factor=0.5)

DENOISE_EPOCHS = 30
train_losses, val_losses = [], []
best_val_loss = float('inf')
best_model_path = 'best_dae.pth'

print(f'Phase 2: Denoising Fine-Tune ({DENOISE_EPOCHS} epochs, σ={NOISE_FACTOR})')
print('-' * 55)

for epoch in range(1, DENOISE_EPOCHS + 1):
    # ---- Train ----
    model.train()
    total_loss = 0.0
    for imgs, _ in train_loader:
        imgs = imgs.to(device)
        noisy_imgs = add_gaussian_noise(imgs, NOISE_FACTOR)  # corrupt input
        optimizer_p2.zero_grad()
        output = model(noisy_imgs)
        loss = criterion(output, imgs)                       # target = clean
        loss.backward()
        optimizer_p2.step()
        total_loss += loss.item() * imgs.size(0)
    train_loss = total_loss / len(train_subset)

    # ---- Validate ----
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for imgs, _ in val_loader:
            imgs = imgs.to(device)
            noisy_imgs = add_gaussian_noise(imgs, NOISE_FACTOR)
            output = model(noisy_imgs)
            val_loss += criterion(output, imgs).item() * imgs.size(0)
    val_loss /= len(val_subset)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    scheduler.step(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), best_model_path)

    if epoch % 5 == 0 or epoch == 1:
        print(f'  Epoch [{epoch:2d}/{DENOISE_EPOCHS}]  Train: {train_loss:.5f}  Val: {val_loss:.5f}')

print(f'\nBest validation loss : {best_val_loss:.5f}')

## 6. Evaluate on Test Set

In [ ]:
# Load best checkpoint
model.load_state_dict(torch.load(best_model_path, map_location=device))
model.eval()

def psnr(clean: torch.Tensor, recon: torch.Tensor) -> float:
    """Peak Signal-to-Noise Ratio (higher = better, in dB)"""
    mse = F.mse_loss(recon, clean).item()
    return 10.0 * np.log10(1.0 / mse)

all_clean, all_noisy, all_recon = [], [], []
test_loss = 0.0

with torch.no_grad():
    for imgs, _ in test_loader:
        imgs = imgs.to(device)
        noisy_imgs = add_gaussian_noise(imgs, NOISE_FACTOR)
        outputs    = model(noisy_imgs)
        test_loss += criterion(outputs, imgs).item() * imgs.size(0)
        all_clean.append(imgs.cpu())
        all_noisy.append(noisy_imgs.cpu())
        all_recon.append(outputs.cpu())

test_loss /= len(test_data)
ac = torch.cat(all_clean)
an = torch.cat(all_noisy)
ar = torch.cat(all_recon)

psnr_noisy    = psnr(ac, an)
psnr_denoised = psnr(ac, ar)

print('=' * 50)
print(f'Test MSE Loss              : {test_loss:.5f}')
print(f'PSNR (noisy vs clean)      : {psnr_noisy:.2f} dB')
print(f'PSNR (denoised vs clean)   : {psnr_denoised:.2f} dB')
print(f'PSNR improvement           : {psnr_denoised - psnr_noisy:+.2f} dB')
print('=' * 50)

## 7. Result Visualization

In [ ]:
# ── Figure 1: Training & Validation Loss Curves ──────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
ep = range(1, len(train_losses) + 1)
ax.plot(ep, train_losses, label='Train Loss', color='#4C9BE8', lw=2, marker='o', ms=4)
ax.plot(ep, val_losses,   label='Val Loss',   color='#E8784C', lw=2, marker='s', ms=4, ls='--')
ax.set_xlabel('Epoch (Phase 2 — Denoising)', fontsize=12)
ax.set_ylabel('MSE Loss', fontsize=12)
ax.set_title('Training & Validation Loss', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('loss_curves.png', dpi=150)
plt.show()

In [ ]:
# ── Figure 2: Clean | Noisy | Denoised Grid ───────────────────────────────────
n = 10
fig, axes = plt.subplots(3, n, figsize=(18, 5.5))
fig.suptitle(
    f'Denoising Autoencoder — Test Set Results\n'
    f'Test MSE = {test_loss:.4f}  |  PSNR Gain: {psnr_denoised - psnr_noisy:+.1f} dB',
    fontsize=12, fontweight='bold', y=1.02
)
for c in range(n):
    axes[0, c].imshow(ac[c].squeeze(), cmap='gray', vmin=0, vmax=1); axes[0, c].axis('off')
    axes[1, c].imshow(an[c].squeeze(), cmap='gray', vmin=0, vmax=1); axes[1, c].axis('off')
    axes[2, c].imshow(ar[c].squeeze(), cmap='gray', vmin=0, vmax=1); axes[2, c].axis('off')

for r, lbl in enumerate(['Clean', f'Noisy (σ={NOISE_FACTOR})', 'Denoised']):
    axes[r, 0].set_ylabel(lbl, fontsize=11, rotation=90, labelpad=48, va='center')

plt.tight_layout()
plt.savefig('comparison_grid.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Figure 3: Noise Robustness Across σ Levels ────────────────────────────────
noise_levels = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]
sample_clean = ac[5:6].to(device)

fig, axes = plt.subplots(3, len(noise_levels), figsize=(16, 5))
fig.suptitle('Denoising Robustness Across Noise Levels', fontsize=13, fontweight='bold')

for i, nf in enumerate(noise_levels):
    noisy_sample = add_gaussian_noise(sample_clean, nf)
    with torch.no_grad():
        denoised_sample = model(noisy_sample)
    p = psnr(sample_clean.cpu(), denoised_sample.cpu())

    axes[0, i].imshow(sample_clean[0].cpu().squeeze(), cmap='gray', vmin=0, vmax=1)
    axes[0, i].axis('off')
    axes[1, i].imshow(noisy_sample[0].cpu().squeeze(), cmap='gray', vmin=0, vmax=1)
    axes[1, i].set_title(f'σ = {nf}', fontsize=10); axes[1, i].axis('off')
    axes[2, i].imshow(denoised_sample[0].cpu().squeeze(), cmap='gray', vmin=0, vmax=1)
    axes[2, i].set_xlabel(f'PSNR: {p:.1f} dB', fontsize=9); axes[2, i].axis('off')

for r, lbl in enumerate(['Clean', 'Noisy', 'Denoised']):
    axes[r, 0].set_ylabel(lbl, fontsize=11)

plt.tight_layout()
plt.savefig('noise_robustness.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Figure 4: Per-Sample Residual Maps ────────────────────────────────────────
fig, axes = plt.subplots(4, 6, figsize=(15, 9))
fig.suptitle('Per-Sample Analysis: Clean | Noisy | Denoised | Residual', fontsize=13, fontweight='bold')

for row, lbl in enumerate(['Clean', 'Noisy', 'Denoised', 'Residual |C−D|']):
    axes[row, 0].set_ylabel(lbl, fontsize=10, rotation=90, labelpad=35, va='center')

for col in range(6):
    idx = col * 15
    cn  = ac[idx].squeeze().numpy()
    no  = an[idx].squeeze().numpy()
    re  = ar[idx].squeeze().numpy()
    res = np.abs(cn - re)

    axes[0, col].imshow(cn,  cmap='gray', vmin=0, vmax=1); axes[0, col].axis('off')
    axes[1, col].imshow(no,  cmap='gray', vmin=0, vmax=1); axes[1, col].axis('off')
    axes[2, col].imshow(re,  cmap='gray', vmin=0, vmax=1); axes[2, col].axis('off')
    im = axes[3, col].imshow(res, cmap='hot', vmin=0, vmax=0.25); axes[3, col].axis('off')

plt.colorbar(im, ax=axes[3, :], fraction=0.02, pad=0.01, label='Absolute Error')
plt.tight_layout()
plt.savefig('residual_map.png', dpi=150, bbox_inches='tight')
plt.show()

---

## 8. Observations & Analysis

### Results Summary

| Metric | Value |
|--------|-------|
| Test MSE (noisy → clean) | 0.02003 |
| PSNR of noisy images | 10.99 dB |
| PSNR of denoised images | 16.98 dB |
| **PSNR Improvement** | **+5.99 dB** |

---

### Key Observations

**1. The model denoise effectively.**  
A +6 dB PSNR gain is a meaningful result. The denoised images clearly recover digit structure that is visually unreadable in the noisy versions — strokes, loops, and endpoints are visible again.

**2. Two-phase training was critical.**  
Without the clean-reconstruction warmup, the model struggled to converge: training loss stayed near the noisy baseline (~0.111) for many epochs. Warming up the weights on clean images first gave the model a good starting point, after which denoising fine-tuning converged rapidly in just 6 epochs.

**3. Convolutional bottleneck forces denoising.**  
The encoder compresses 28×28 → 7×7 with only 8 channels (392 values). This spatial compression forces the model to learn a compact digit representation and discard random, high-frequency noise which cannot be encoded efficiently into the bottleneck.

**4. Noise robustness is graceful but bounded.**  
From the robustness figure:
- σ ≤ 0.3: near-perfect denoising, digit shape fully preserved
- σ = 0.4 (training level): good denoising, visible PSNR ~17 dB  
- σ ≥ 0.6: denoised output blurs — the model reverts to a mean estimate

This is expected: the model generalizes to moderate out-of-distribution noise but fails for severe corruption it never saw.

**5. Residual maps reveal structure-dependent error.**  
The residual heatmaps (|Clean − Denoised|) show the largest errors at stroke edges and in complex digit strokes (e.g., digits 8 and 3). Background pixels are reconstructed almost perfectly. This confirms the model learned to distinguish signal structure from flat background.

**6. MSE as a loss — advantages and limitations.**  
MSE is differentiable and well-suited for reconstruction tasks on normalized pixel data. However, MSE over-penalizes large pixel errors and can produce slightly blurry reconstructions. A Structural Similarity (SSIM) loss or a perceptual loss would likely yield sharper digit strokes.

---

### Challenges

- **Convergence without warmup:** Directly training with noisy inputs caused the model to output near-mean images (high MSE plateau). The warmup phase bypassed this.
- **CPU training speed:** Full 60k dataset training on CPU is slow (~100 seconds per epoch with batch=512). Using a 10k subset made rapid experimentation feasible while still producing strong metrics.

---

### Potential Improvements

| Idea | Expected Benefit |
|------|------------------|
| BatchNorm in all encoder/decoder blocks | More stable training, faster convergence |
| Skip connections (U-Net style) | Preserve fine details, reduce blur |
| SSIM or Perceptual loss | Sharper edges, visually better output |
| Noise annealing (start small, increase σ) | Better convergence at high noise |
| Deeper architecture (more channels) | Higher capacity → lower test MSE |
| Train on full 60k dataset | Better generalization |
